# 🩺 Diabetic Retinopathy Detection — Deep Learning Training Notebook
### **Diabetic Vision Health Management System**

This notebook trains a **Deep Convolutional Neural Network (ResNet-50 / EfficientNet)** on retinal fundus imagery using Transfer Learning to predict Diabetic Retinopathy severity across the 5 standard ICDR grades:
- **Grade 0**: No Diabetic Retinopathy
- **Grade 1**: Mild Non-Proliferative DR (NPDR)
- **Grade 2**: Moderate NPDR
- **Grade 3**: Severe NPDR
- **Grade 4**: Proliferative Diabetic Retinopathy (PDR)

> **GPU Recommended**: Go to `Runtime` -> `Change runtime type` -> Select `T4 GPU`.

In [ ]:
# 1. Install & Verify Environment
!pip install -q torch torchvision torchaudio scikit-learn matplotlib pandas pillow

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Import Core Libraries
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, cohen_kappa_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 3. Data Preprocessing & Augmentation
# Circular cropping, resizing to 224x224, and ImageNet normalization

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# 4. Define Transfer Learning Model Architecture
def build_model(num_classes=5):
    # Pretrained ResNet-50
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    
    # Fine-tune the higher-level feature layers
    for param in list(model.parameters())[:-15]:
        param.requires_grad = False
        
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model

model = build_model(num_classes=5).to(device)
print(model.fc)

In [ ]:
# 5. Training Setup: Loss, Optimizer & Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

In [ ]:
# 6. Save Model Checkpoint for DiaVision System
# After training finishes, save the weights:
torch.save(model.state_dict(), 'dr_model_resnet50.pth')
print("Model saved! Download 'dr_model_resnet50.pth' and place in C:\\Capstone\\ml-service\\models/")